## 面试问题

确定性重放：固定模型输出与观察后让整条 loop 可复现？

## 回答主线

给定相同的观察/模型输出序列，纯函数 + 幂等的循环应产出完全相同的状态轨迹。本 Notebook 记录一次运行的观察序列，用它重放同一纯函数循环，验证轨迹逐步复现；再用一个内部读递增计数器的不纯循环，展示同一序列重放出不同轨迹、无法复现。

## 真实案例

记录的观察序列 `[10, 20, 30]`。纯循环逐步累加得轨迹 `[10, 30, 60]`，重放两次一致；不纯循环每步混入递增计数器，重放两次不同。数据为教学序列，不代表真实模型。

In [1]:
recorded_observations = [10, 20, 30]  # 一次运行记录下来的观察序列。
print("记录的观察序列:", recorded_observations)  # 展示用于重放的输入序列。
print("重放将用同一序列驱动同一循环并逐步比对状态")  # 说明重放思路。

记录的观察序列: [10, 20, 30]
重放将用同一序列驱动同一循环并逐步比对状态


## 基线（Baseline）

先建立可重放的基线：纯函数步只依赖注入的 state 和 obs，运行并记录逐步轨迹作为「原始轨迹」。

In [2]:
def pure_step(state, obs):  # 纯函数步：只依赖注入的 state 与 obs。
    return {"total": state["total"] + obs}  # 累加观察产出新状态。

def run_and_record(observations):  # 运行纯循环并记录逐步轨迹。
    state = {"total": 0}  # 初始状态。
    trajectory = []  # 逐步状态轨迹。
    for obs in observations:  # 逐个注入观察。
        state = pure_step(state, obs)  # 执行纯步。
        trajectory.append(state["total"])  # 记录该步状态。
    return trajectory  # 返回状态轨迹。

original = run_and_record(recorded_observations)  # 原始运行的轨迹。
print("原始轨迹:", original)  # 展示原始逐步状态。

原始轨迹: [10, 30, 60]


## 失败案例与修正

先看重放成功：用记录序列重放纯循环两次，轨迹都等于原始。再看失败：不纯循环内部读递增计数器，同一序列重放两次得到不同轨迹，无法复现。

In [3]:
replay1 = run_and_record(recorded_observations)  # 用记录序列重放一次。
replay2 = run_and_record(recorded_observations)  # 再重放一次。
print("重放轨迹1:", replay1)  # 展示第一次重放。
print("重放轨迹2:", replay2)  # 展示第二次重放。
print("重放是否逐步等于原始:", replay1 == original and replay2 == original)  # 展示纯循环重放确定复现。

重放轨迹1: [10, 30, 60]
重放轨迹2: [10, 30, 60]
重放是否逐步等于原始: True


In [4]:
drift = {"counter": 0}  # 不纯步依赖的外部计数器。

def impure_step(state, obs):  # 不纯步：内部读取变化的计数器。
    drift["counter"] += 1  # 每步改变外部状态。
    return {"total": state["total"] + obs + drift["counter"]}  # 结果混入不可注入的漂移。

def run_impure(observations):  # 运行不纯循环记录轨迹。
    state = {"total": 0}  # 初始状态。
    trajectory = []  # 逐步轨迹。
    for obs in observations:  # 逐个注入观察。
        state = impure_step(state, obs)  # 执行不纯步。
        trajectory.append(state["total"])  # 记录该步状态。
    return trajectory  # 返回轨迹。

impure1 = run_impure(recorded_observations)  # 第一次运行不纯循环。
impure2 = run_impure(recorded_observations)  # 第二次运行不纯循环。
print("不纯轨迹1:", impure1)  # 展示第一次。
print("不纯轨迹2:", impure2)  # 展示第二次因计数器漂移不同。

不纯轨迹1: [11, 33, 66]
不纯轨迹2: [14, 39, 75]


In [5]:
print("纯循环重放一致:", replay1 == original and replay2 == original)  # 纯循环可确定重放。
print("不纯循环重放一致:", impure1 == impure2)  # 不纯循环不可重放。
print("原始轨迹:", original, "| 纯循环重放:", replay1)  # 展示纯循环轨迹一致。
print("不纯两次轨迹:", impure1, "vs", impure2)  # 展示不纯两次不同。

纯循环重放一致: True
不纯循环重放一致: False
原始轨迹: [10, 30, 60] | 纯循环重放: [10, 30, 60]
不纯两次轨迹: [11, 33, 66] vs [14, 39, 75]


## 结果解读

纯循环用记录序列重放两次，轨迹都逐步等于原始 `[10, 30, 60]`——可确定复现；不纯循环因内部计数器跨运行漂移，两次重放轨迹不同。要点：重放依赖纯函数 + 幂等、比对整条轨迹、记录输入而非输出、重放是回归的基础。

In [6]:
assert original == [10, 30, 60]  # 原始轨迹符合预期。
assert replay1 == original  # 纯循环重放逐步复现原始轨迹。
assert replay2 == original  # 再次重放仍复现。
assert impure1 != impure2  # 不纯循环两次重放不一致。
print("全部不变量通过")  # 输出测试通过信号。

全部不变量通过
